# Protocolli 3.1 — Report completo da output ruoli

Questo notebook legge un JSON già arricchito con:

- match RUNTS
- eventuale verifica enti sovracomunali
- campo `ruolo` dentro ogni soggetto

e genera:

- `report_sintetico.txt`
- `tipologie_numerosita.csv`
- `attori_coinvolti.csv`
- `soggetti_gestione_monitoraggio.csv`
- `soggetti_per_file.csv`
- `soggetti_proponenti.csv`
- `firmatari_per_file.csv`
- `rete_attori_protocollo.csv`
- `network_attori.graphml`
- `report_strutturato.json`


In [2]:
from pathlib import Path
import json
import csv
import re
import itertools
from collections import Counter, defaultdict
import networkx as nx

# =========================================================
# CONFIG
# =========================================================


reg_code = "09"   # es. "08" Emilia-Romagna

JSON_STEP1_FOLDER = f"output/json/step_1/"
INPUT_JSON = Path(JSON_STEP1_FOLDER) / f"{reg_code}_risultati_enriched_2.4.json"

OUT_DIR = Path(f"output/reports/{reg_code}_report_output")
OUT_DIR.mkdir(exist_ok=True)

REPORT_TXT = OUT_DIR / "report_sintetico.txt"
TIPI_CSV = OUT_DIR / "tipologie_numerosita.csv"
ATTORI_CSV = OUT_DIR / "attori_coinvolti.csv"
GESTIONE_CSV = OUT_DIR / "soggetti_gestione_monitoraggio.csv"
SOGGETTI_PER_FILE_CSV = OUT_DIR / "soggetti_per_file.csv"
PROPONENTI_CSV = OUT_DIR / "soggetti_proponenti.csv"
FIRMATARI_CSV = OUT_DIR / "firmatari_per_file.csv"
RETE_CSV = OUT_DIR / "rete_attori_protocollo.csv"
GRAPHML_FILE = OUT_DIR / "network_attori.graphml"
REPORT_JSON = OUT_DIR / "report_strutturato.json"

print("INPUT_JSON:", INPUT_JSON)
print("OUT_DIR:", OUT_DIR)


INPUT_JSON: output\json\step_1\09_risultati_enriched_2.4.json
OUT_DIR: output\reports\09_report_output


In [3]:
def get_entities_list_ref(file_item: dict):
    '''
    Restituisce il riferimento alla lista entità/soggetti da arricchire.
    Supporta:
    - vecchio formato: file_item["risultato"]["entities"]
    - nuovo formato:   file_item["soggetti"]
    '''
    if isinstance(file_item.get("soggetti"), list):
        return file_item["soggetti"]
    risultato = file_item.get("risultato") or {}
    if isinstance(risultato.get("entities"), list):
        return risultato["entities"]
    return []



# =========================================================
# HELPERS
# =========================================================
def clean(s):
    return str(s or "").strip()

def norm(s):
    return re.sub(r"\s+", " ", clean(s)).strip().upper()

def unique_entities_by_name(data):
    by_name = {}

    for file_item in data:
        file_name = file_item.get("file", "")
        #entities = (file_item.get("risultato") or {}).get("entities") or []

        entities = get_entities_list_ref(file_item)

        

        for ent in entities:
            nome = clean(ent.get("nome"))
            if not nome:
                continue

            k = norm(nome)

            if k not in by_name:
                by_name[k] = dict(ent)
                by_name[k]["files"] = {file_name} if file_name else set()
            else:
                if file_name:
                    by_name[k]["files"].add(file_name)

                for field in [
                    "tipo", "comune", "indirizzo", "ente_capofila", "note",
                    "cf", "comune_runts", "natura_runts", "denominazione_runts"
                ]:
                    if not clean(by_name[k].get(field)) and clean(ent.get(field)):
                        by_name[k][field] = ent.get(field)

    for k in by_name:
        by_name[k]["files"] = sorted(list(by_name[k]["files"]))

    return list(by_name.values())

def macro_categoria(ent):
    tipo = clean(ent.get("tipo"))

    if tipo in {"CAV/Centri Antiviolenza", "Case Rifugio"}:
        return "CAV e Case Rifugio"

    if tipo in {
        "Ente terzo settore - ETS (iscritto al RUNTS)",
        "Ente terzo settore - ETS (iscritto al RUNTS) costituito da donne per le donne",
        "Associazioni che si occupano di programmi di prevenzione, recupero e trattamento per uomini maltrattanti",
    }:
        return "Associazioni / ETS"

    if tipo in {
        "Comuni",
        "Province/Città metropolitane",
        "Regioni/Province Autonome",
        "Unione dei Comuni",
        "Comunità di montagna",
        "Comunità territoriali del Trentino",
        "Ambiti della programmazione sociale e socio-sanitaria (Ambiti Sociali, Piani di Zona, Distretti socio-sanitari, Società della Salute)",
        "Servizi sociali comunali",
        "Settore educativo comunale",
        "Servizio abusi e maltrattamenti comunale",
        "Polizia Municipale",
    }:
        return "Enti territoriali / sovracomunali / servizi comunali"

    if tipo in {
        "ASL (consultori familiari e altri servizi territoriali)",
        "Ospedale (Pronto soccorso, ecc.)",
        "Servizi per l'impiego",
    }:
        return "Sanità e servizi territoriali"

    if tipo in {
        "Prefettura",
        "Questura",
        "Carabinieri/Polizia/altre forze dell'ordine",
        "Procura Minorile/Tribunale minorile",
        "Procura Ordinaria/Tribunale/Corte d'appello",
    }:
        return "Giustizia e forze dell'ordine"

    if tipo in {
        "Ordine avvocati",
        "Ordine psicologi e Ordine assistenti sociali",
        "Ordine medici e odontoiatri e Ordine farmacisti",
        "Altri ordini professionali (infermieri, ostetriche, giornalisti, commercialisti, ecc.)",
        "Sindacati/Associazioni di categoria",
        "Università",
        "Scuole/Ufficio scolastico provinciale e regionale",
        "Organismi di parità",
    }:
        return "Altri attori istituzionali / professionali"

    return "Altro"

def extract_ruoli(ent):
    note = clean(ent.get("note"))
    capofila = clean(ent.get("ente_capofila"))
    text = f"{capofila} {note}".upper()

    ruoli = set()

    if any(x in text for x in [
        "GESTORE", "GESTISCE", "GESTIONE", "ENTE GESTORE",
        "SOGGETTO GESTORE", "SERVIZIO GESTITO", "INCARICATO DELLA GESTIONE"
    ]):
        ruoli.add("gestione")

    if any(x in text for x in [
        "COORDINA", "COORDINAMENTO", "TAVOLO DI COORDINAMENTO",
        "COORDINARE", "PRESIEDE IL TAVOLO", "ENTE CAPOFILA", "CAPOFILA",
        "CABINA DI REGIA", "GOVERNANCE"
    ]):
        ruoli.add("coordinamento")

    if any(x in text for x in [
        "MONITORA", "MONITORAGGIO", "VERIFICA", "OSSERVATORIO"
    ]):
        ruoli.add("monitoraggio")

    return sorted(ruoli)

def choose_soggetto_incaricato(ent):
    ruoli = extract_ruoli(ent)
    if not ruoli:
        return ""

    ente_capofila = clean(ent.get("ente_capofila"))
    if ente_capofila:
        return ente_capofila

    return clean(ent.get("nome"))


def get_document_roles(ent):
    """
    Restituisce i ruoli documentali già presenti nell'entità.
    Valori ammessi attesi:
    - firmatario
    - soggetto_proponente
    - attore
    """
    ruolo = ent.get("ruolo") or []
    if isinstance(ruolo, str):
        ruolo = [ruolo]
    validi = {"firmatario", "soggetto_proponente", "attore"}
    out = []
    for r in ruolo:
        rr = clean(r)
        if rr in validi and rr not in out:
            out.append(rr)
    return out


In [4]:
# =========================================================
# BUILD REPORT
# =========================================================
def build_report(data):
    entities = unique_entities_by_name(data)

    tipi_counter = Counter()
    for ent in entities:
        tipo = clean(ent.get("tipo")) or "Altro"
        tipi_counter[tipo] += 1

    tipologie_numerosita = [
        {"tipo": tipo, "numerosita": count}
        for tipo, count in tipi_counter.most_common()
    ]

    macro_groups = defaultdict(list)
    macro_files = defaultdict(set)

    for ent in entities:
        macro = macro_categoria(ent)
        nome = clean(ent.get("nome"))
        if nome:
            macro_groups[macro].append(nome)
        for f in ent.get("files", []):
            if f:
                macro_files[macro].add(f)

    attori_coinvolti = []
    for macro, nomi in macro_groups.items():
        nomi_unici = sorted(set(x for x in nomi if x))
        attori_coinvolti.append({
            "macro_categoria": macro,
            "numerosita": len(nomi_unici),
            "attori": nomi_unici,
            "files": sorted(macro_files.get(macro, set()))
        })

    attori_coinvolti.sort(key=lambda x: (-x["numerosita"], x["macro_categoria"]))

    soggetti_gestione = []
    seen = set()

    for ent in entities:
        ruoli = extract_ruoli(ent)
        if not ruoli:
            continue

        soggetto = choose_soggetto_incaricato(ent)
        nome = clean(ent.get("nome"))
        tipo = clean(ent.get("tipo"))
        note = clean(ent.get("note"))
        ente_capofila = clean(ent.get("ente_capofila"))
        files = ent.get("files", [])

        key = (norm(soggetto), norm(nome), "|".join(ruoli))
        if key in seen:
            continue
        seen.add(key)

        soggetti_gestione.append({
            "soggetto_incaricato": soggetto,
            "nome_entita": nome,
            "tipo_entita": tipo,
            "ruoli": ruoli,
            "ente_capofila": ente_capofila,
            "note": note,
            "files": files
        })

    soggetti_gestione.sort(key=lambda x: (x["soggetto_incaricato"], x["nome_entita"]))

    proponenti_rows = []
    firmatari_rows = []

    for file_item in data:
        file_name = clean(file_item.get("file"))

        for p in file_item.get("soggetti_proponenti", []) or []:
            p = clean(p)
            if p:
                proponenti_rows.append({
                    "file": file_name,
                    "soggetto_proponente": p
                })

        for f in file_item.get("firmatari", []) or []:
            f = clean(f)
            if f:
                firmatari_rows.append({
                    "file": file_name,
                    "firmatario": f
                })

    report = {
        "input_json": str(INPUT_JSON),
        "totale_soggetti_unici": len(entities),
        "tipologia_numerosita": tipologie_numerosita,
        "attori_coinvolti": attori_coinvolti,
        "soggetti_incaricati_gestione_monitoraggio_coordinamento": soggetti_gestione,
        "soggetti_proponenti": proponenti_rows,
        "firmatari": firmatari_rows
    }

    return report


In [5]:
# =========================================================
# EXPORT
# =========================================================
def save_csv_tipologie(rows, path):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["tipo", "numerosita"])
        w.writeheader()
        w.writerows(rows)

def save_csv_attori(rows, path):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=["macro_categoria", "numerosita", "attori", "files"]
        )
        w.writeheader()

        for r in rows:
            rr = dict(r)
            rr["attori"] = " | ".join(r["attori"])
            rr["files"] = " | ".join(sorted(set(r.get("files", []))))
            w.writerow(rr)

def save_csv_gestione(rows, path):
    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(
            f,
            fieldnames=[
                "soggetto_incaricato",
                "nome_entita",
                "tipo_entita",
                "ruoli",
                "ente_capofila",
                "note",
                "files"
            ]
        )

        w.writeheader()

        for r in rows:
            rr = dict(r)
            rr["ruoli"] = ", ".join(r["ruoli"])
            rr["files"] = " | ".join(r.get("files", []))
            w.writerow(rr)

def save_csv_soggetti_per_file(data, path):
    rows = []

    for file_item in data:
        file_name = clean(file_item.get("file", ""))
        entities = get_entities_list_ref(file_item)

        for ent in entities:
            nome = clean(ent.get("nome"))
            tipo = clean(ent.get("tipo"))
            ente_capofila = clean(ent.get("ente_capofila"))
            note = clean(ent.get("note")).lower()

            ruoli_doc = get_document_roles(ent)

            ruoli_funzione = []
            if "gest" in note:
                ruoli_funzione.append("gestione")
            if "monitor" in note or "verifica" in note or "osservatorio" in note:
                ruoli_funzione.append("monitoraggio")
            if "coord" in note or "cabina di regia" in note or "governance" in note:
                ruoli_funzione.append("coordinamento")

            rows.append({
                "file": file_name,
                "nome_entita": nome,
                "tipo": tipo,
                "ruolo_documentale": ", ".join(ruoli_doc),
                "ruolo_funzione": ", ".join(ruoli_funzione),
                "ente_capofila": ente_capofila
            })

    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "file", "nome_entita", "tipo",
                "ruolo_documentale", "ruolo_funzione", "ente_capofila"
            ]
        )
        writer.writeheader()
        writer.writerows(rows)

def save_csv_proponenti(rows, path):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "soggetto_proponente"])
        writer.writeheader()
        writer.writerows(rows)

def save_csv_firmatari(rows, path):
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "firmatario"])
        writer.writeheader()
        writer.writerows(rows)

def save_rete_attori_per_file(data, csv_path, graphml_path):
    edge_rows = []
    edge_counter = Counter()
    node_counter = Counter()

    for file_item in data:
        file_name = clean(file_item.get("file"))
        entities = get_entities_list_ref(file_item)

        attori = []
        for ent in entities:
            nome = clean(ent.get("nome"))
            ruoli_doc = get_document_roles(ent)
            if nome and ("attore" in ruoli_doc or not ruoli_doc):
                attori.append(nome)

        attori_unici = sorted(set(attori))

        for attore in attori_unici:
            node_counter[attore] += 1

        for a1, a2 in itertools.combinations(attori_unici, 2):
            edge_rows.append({
                "file": file_name,
                "attore_1": a1,
                "attore_2": a2
            })
            key = tuple(sorted([a1, a2]))
            edge_counter[key] += 1

    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["file", "attore_1", "attore_2"])
        writer.writeheader()
        writer.writerows(edge_rows)

    G = nx.Graph()

    for attore, file_count in node_counter.items():
        G.add_node(attore, label=attore, n_files=file_count)

    for (a1, a2), weight in edge_counter.items():
        G.add_edge(a1, a2, weight=weight)

    nx.write_graphml(G, graphml_path)

def save_txt_report(report, path):
    lines = []

    lines.append("REPORT SOGGETTI\n")
    lines.append(f"Input JSON: {report['input_json']}")
    lines.append(f"Totale soggetti unici: {report['totale_soggetti_unici']}\n")

    lines.append("\n1) TIPOLOGIA E NUMEROSITA DEI SOGGETTI\n")
    for row in report["tipologia_numerosita"]:
        lines.append(f"- {row['tipo']}: {row['numerosita']}")

    lines.append("\n\n2) ATTORI COINVOLTI\n")
    for row in report["attori_coinvolti"]:
        lines.append(f"\n{row['macro_categoria']} ({row['numerosita']}):")
        if row.get("files"):
            lines.append(f"  file: {', '.join(row.get('files', []))}")
        for nome in row["attori"]:
            lines.append(f"  - {nome}")

    lines.append("\n\n3) SOGGETTI INCARICATI DI GESTIRE / MONITORARE / COORDINARE\n")
    if not report["soggetti_incaricati_gestione_monitoraggio_coordinamento"]:
        lines.append("- Nessun soggetto rilevato con questa logica.")
    else:
        for row in report["soggetti_incaricati_gestione_monitoraggio_coordinamento"]:
            ruolo_txt = ", ".join(row["ruoli"])
            lines.append(
                f"- soggetto_incaricato: {row['soggetto_incaricato']} | "
                f"entita: {row['nome_entita']} | "
                f"tipo: {row['tipo_entita']} | "
                f"ruoli: {ruolo_txt} | "
                f"files: {', '.join(row.get('files', []))}"
            )

    lines.append("\n\n4) SOGGETTI PROPONENTI\n")
    if not report["soggetti_proponenti"]:
        lines.append("- Nessun soggetto proponente rilevato.")
    else:
        for row in report["soggetti_proponenti"]:
            lines.append(f"- file: {row['file']} | soggetto_proponente: {row['soggetto_proponente']}")

    lines.append("\n\n5) FIRMATARI\n")
    if not report["firmatari"]:
        lines.append("- Nessun firmatario rilevato.")
    else:
        for row in report["firmatari"]:
            lines.append(f"- file: {row['file']} | firmatario: {row['firmatario']}")

    path.write_text("\n".join(lines), encoding="utf-8")


In [6]:
# =========================================================
# RUN
# =========================================================
if not INPUT_JSON.exists():
    raise FileNotFoundError(f"File non trovato: {INPUT_JSON}")

data = json.loads(INPUT_JSON.read_text(encoding="utf-8"))
report = build_report(data)

REPORT_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
save_csv_tipologie(report["tipologia_numerosita"], TIPI_CSV)
save_csv_attori(report["attori_coinvolti"], ATTORI_CSV)
save_csv_gestione(report["soggetti_incaricati_gestione_monitoraggio_coordinamento"], GESTIONE_CSV)
save_csv_soggetti_per_file(data, SOGGETTI_PER_FILE_CSV)
save_csv_proponenti(report["soggetti_proponenti"], PROPONENTI_CSV)
save_csv_firmatari(report["firmatari"], FIRMATARI_CSV)
save_rete_attori_per_file(data, RETE_CSV, GRAPHML_FILE)
save_txt_report(report, REPORT_TXT)

print("✅ Report creato da output ruoli")
print("📄", REPORT_TXT)
print("📄", TIPI_CSV)
print("📄", ATTORI_CSV)
print("📄", GESTIONE_CSV)
print("📄", SOGGETTI_PER_FILE_CSV)
print("📄", PROPONENTI_CSV)
print("📄", FIRMATARI_CSV)
print("📄", RETE_CSV)
print("📄", GRAPHML_FILE)
print("📄", REPORT_JSON)


✅ Report creato da output ruoli
📄 output\reports\09_report_output\report_sintetico.txt
📄 output\reports\09_report_output\tipologie_numerosita.csv
📄 output\reports\09_report_output\attori_coinvolti.csv
📄 output\reports\09_report_output\soggetti_gestione_monitoraggio.csv
📄 output\reports\09_report_output\soggetti_per_file.csv
📄 output\reports\09_report_output\soggetti_proponenti.csv
📄 output\reports\09_report_output\firmatari_per_file.csv
📄 output\reports\09_report_output\rete_attori_protocollo.csv
📄 output\reports\09_report_output\network_attori.graphml
📄 output\reports\09_report_output\report_strutturato.json
